In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
import time 
from functools import partial
from pyscf import gto, scf, fci
from jax import flatten_util
from itertools import combinations
import itertools

from NES_VMC_H2_631G import get_ccsd_excitations_and_sampler_edges_from_hf,\
SingleStateAnsatz,create_machine,compute_local_energies,forces_expect_hermitian,compute_qgt


/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: uv is a replacement for pip which helps you follow good software practices.

E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV


In [3]:
bond_length = 1.8
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='6-31G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)
hf_ground_energy = mf.e_tot
print(f'HF 基准能量: {hf_ground_energy:.8f}')

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=4,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
ha = nkx.operator.from_pyscf_molecule(mol)
Hatree_Fock = hi.all_states()[0]

# ============================================================
# 2. Sampler edges
# ============================================================

Hatree_Fock = hi.all_states()[0]
alpha_orbs = [0, 1, 2, 3]
beta_orbs  = [4, 5, 6, 7]

single_edges_full = (
    list(itertools.combinations(alpha_orbs, 2))
    + list(itertools.combinations(beta_orbs, 2))
)
g = nk.graph.Graph(edges=single_edges_full)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=100, sweep_size=32)



HF 基准能量: -0.94605220
H₂ FCI 基准能量
E0 = -1.02613572 Ha  |  激发能: 0.0000 eV
E1 = -0.97892204 Ha  |  激发能: 1.2848 eV
E2 = -0.66776157 Ha  |  激发能: 9.7519 eV
E3 = -0.60817046 Ha  |  激发能: 11.3734 eV


In [5]:
# ===================== 6. 初始化 =====================
rngs = nnx.Rngs(21)
# 减小隐藏层维度，避免参数过多导致数值不稳定
model = SingleStateAnsatz(hi.size, hidden_dim=16, rngs=rngs)
machine, graphdef, params = create_machine(model)
sampler_state = sampler.init_state(machine, params, seed=1)

# 关键：自然梯度尺度远大于普通梯度，必须用极小的学习率
# 或者使用更稳定的优化器如 Adam
optimizer = optax.adam(learning_rate=0.001)  # 用 Adam 代替 SGD
# optimizer = optax.sgd(learning_rate=0.0001)  # 如果用 SGD，学习率要到 0.0001 甚至更小
opt_state = optimizer.init(params)

# 训练参数
N_ITER = 1000  # 迭代次数
N_SAMPLES = 1008  # 样本数

# ===================== 7. 训练循环（带诊断指标）=====================
print("\n" + "="*90)
print("开始纯 JAX VMC 训练 (自然梯度下降法) - 带诊断指标")
print("="*90)
print(f"{'Step':>5} | {'E':>12} | {'±':>8} | {'Error':>8} | {'‖∇E‖':>10} | {'‖∇nat‖':>10} | {'cond(QGT)':>12} | {'‖params‖':>10}")
print("-"*90)

# 用于记录训练历史
history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': [],
    'grad_norm': [],
    'nat_grad_norm': [],
    'qgt_cond': [],
    'params_norm': []
}

for step in range(N_ITER):
    # 1. 采样
    sampler_state = sampler.reset(machine, params, sampler_state)
    
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, 
        chain_length=20
    )
    samples = samples.reshape(-1, hi.size)
    
    # 2. 计算 force-based 能量和梯度
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    
    # 计算原始梯度范数（在乘以2之前）
    grad_flat_raw, _ = flatten_util.ravel_pytree(grad)
    grad_norm = float(jnp.sqrt(jnp.sum(jnp.abs(grad_flat_raw)**2)))
    
    # 梯度缩放因子
    grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    
    # 3. 计算 QGT（带诊断）
    qgt_reg, qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.1)
    
    # 计算 QGT 条件数（使用 SVD 的最大/最小奇异值比）
    qgt_for_svd = jnp.real(qgt_reg)  # 取实部做 SVD
    s = jnp.linalg.svd(qgt_for_svd, compute_uv=False)
    qgt_cond = float(s[0] / (s[-1] + 1e-10))  # 避免除以零
    
    grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grad)
  
    # 4. 自然梯度 natural-gradient = S^{-1} * grad（使用更稳定的 solve）
    try:
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        natural_grad = grad_unravel_fn(natural_grad_flat)
        nat_grad_norm = float(jnp.sqrt(jnp.sum(jnp.abs(natural_grad_flat)**2)))
    except Exception as e:
        # 如果 solve 失败，使用 SVD 求逆作为后备
        print(f"  [WARNING] solve failed at step {step}: {e}, using SVD fallback")
        U, s_svd, Vh = jnp.linalg.svd(qgt_reg, full_matrices=False)
        sinv = 1.0 / (s_svd + 0.01)  # 加正则化
        natural_grad_flat = Vh.T @ (sinv[:, None] * (U.T @ grad_flat))
        natural_grad = grad_unravel_fn(natural_grad_flat)
        nat_grad_norm = float(jnp.sqrt(jnp.sum(jnp.abs(natural_grad_flat)**2)))
    
    grad = natural_grad
    
    # 5. 更新参数（自然梯度下降）
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 6. 计算参数范数
    params_flat, _ = flatten_util.ravel_pytree(params)
    params_norm = float(jnp.sqrt(jnp.sum(jnp.abs(params_flat)**2)))
    
    # 7. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        history['grad_norm'].append(grad_norm)
        history['nat_grad_norm'].append(nat_grad_norm)
        history['qgt_cond'].append(qgt_cond)
        history['params_norm'].append(params_norm)
        
        # 格式化输出
        E_str = f"{energy.real:.8f}" if not jnp.isnan(energy.real) else "nan"
        std_str = f"{energy_std:.6f}" if not jnp.isnan(energy_std) else "nan"
        err_str = f"{float(error):.6f}" if not jnp.isnan(error) else "nan"
        gn_str = f"{grad_norm:.6f}"
        ngn_str = f"{nat_grad_norm:.6f}"
        cond_str = f"{qgt_cond:.2e}"
        pn_str = f"{params_norm:.4f}"
        
        print(f"{step:>5} | {E_str:>12} | {std_str:>8} | {err_str:>8} | {gn_str:>10} | {ngn_str:>10} | {cond_str:>12} | {pn_str:>10}")

# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*90)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*90)

# 打印诊断摘要
print("\n" + "="*90)
print("诊断摘要")
print("="*90)
print(f"QGT 条件数范围: {min(history['qgt_cond']):.2e} ~ {max(history['qgt_cond']):.2e}")
print(f"梯度范数范围:   {min(history['grad_norm']):.6f} ~ {max(history['grad_norm']):.6f}")
print(f"自然梯度范数:   {min(history['nat_grad_norm']):.6f} ~ {max(history['nat_grad_norm']):.6f}")
print(f"参数范数范围:   {min(history['params_norm']):.4f} ~ {max(history['params_norm']):.4f}")
print("="*90)



开始纯 JAX VMC 训练 (自然梯度下降法) - 带诊断指标
 Step |            E |        ± |    Error |       ‖∇E‖ |     ‖∇nat‖ |    cond(QGT) |   ‖params‖
------------------------------------------------------------------------------------------
    0 |   0.59820470 | 0.018779 | 1.624340 |   0.906577 |  13.691247 |     1.20e+01 |     5.7728
   50 |  -0.58716760 | 0.011733 | 0.438968 |   0.725577 |   5.344931 |     2.34e+01 |     5.9168
  100 |  -0.97076499 | 0.004822 | 0.055371 |   2.970471 |   1.114545 |     2.45e+02 |     5.9331
  150 |  -0.98957904 | 0.003180 | 0.036557 |   0.450455 |   0.203843 |     5.24e+02 |     5.9349
  200 |  -0.99005372 | 0.003421 | 0.036082 |   0.300045 |   0.113982 |     4.27e+02 |     5.9350
  250 |  -0.99765231 | 0.003215 | 0.028483 |   0.627795 |   0.178561 |     4.93e+02 |     5.9353
  300 |  -0.99758326 | 0.003291 | 0.028552 |   0.342621 |   0.109741 |     4.71e+02 |     5.9357
  350 |  -1.00109691 | 0.003211 | 0.025039 |   0.368980 |   0.105082 |     5.37e+02 |     5.9362
  